In [ ]:
from google.colab import files
from PIL import Image

uploaded = files.upload()  # Choose ONE .pgm file from BOSSbase_1.01
filename = next(iter(uploaded))

with Image.open(filename) as image:
    print("File:", filename)
    print("Size:", image.size)
    print("Mode:", image.mode)

Saving 10.pgm to 10.pgm
File: 10.pgm
Size: (512, 512)
Mode: L


In [ ]:
from google.colab import files

uploaded = files.upload()
print(list(uploaded))

Saving BOSSbase_project_data.zip to BOSSbase_project_data.zip
['BOSSbase_project_data.zip']


In [ ]:
from pathlib import Path
from zipfile import ZipFile

zip_path = Path("/content/BOSSbase_project_data.zip")
project = Path("/content/boss_project")

assert zip_path.is_file(), "ZIP not found in Colab"

with ZipFile(zip_path) as archive:
    archive.extractall(project)

originals = list((project / "BOSSbase_selected_500").glob("*.pgm"))
stego_01 = list((project / "BOSSbase_generated_500" / "0p1").glob("*.pgm"))
stego_04 = list((project / "BOSSbase_generated_500" / "0p4").glob("*.pgm"))
splits = project / "BOSSbase_generated_500" / "splits.csv"

print("Originals:", len(originals))
print("0.1 bpp:", len(stego_01))
print("0.4 bpp:", len(stego_04))
print("Split file exists:", splits.is_file())

Originals: 500
0.1 bpp: 500
0.4 bpp: 500
Split file exists: True


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import csv
import numpy as np
import torch

project = Path("/content/boss_project")
original_folder = project / "BOSSbase_selected_500"
generated_folder = project / "BOSSbase_generated_500"

with (generated_folder / "splits.csv").open(newline="") as file:
    source_rows = list(csv.DictReader(file))

class ImagePairDataset(Dataset):
    def __init__(self, split, payload):
        self.items = []

        for row in source_rows:
            if row["split"] != split:
                continue

            name = row["source_filename"]
            original = original_folder / name
            stego = generated_folder / payload / name

            assert original.is_file() and stego.is_file(), name
            self.items.append((original, 0))
            self.items.append((stego, 1))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]
        with Image.open(path) as image:
            pixels = np.array(image, dtype=np.float32)

        tensor = torch.from_numpy(pixels).unsqueeze(0)
        return tensor, label

train_data = ImagePairDataset("train", "0p4")
val_data = ImagePairDataset("validation", "0p4")
test_data = ImagePairDataset("test", "0p4")

train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4, shuffle=False)

images, labels = next(iter(train_loader))

print("Train images:", len(train_data))
print("Validation images:", len(val_data))
print("Test images:", len(test_data))
print("One batch shape:", tuple(images.shape))
print("Batch labels:", labels.tolist())

Train images: 600
Validation images: 200
Test images: 200
One batch shape: (4, 1, 512, 512)
Batch labels: [1, 0, 0, 1]


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SmallSteganalysisCNN(nn.Module):
    def __init__(self):
        super().__init__()

        kernel = torch.tensor(
            [[0, -1, 0],
             [-1, 4, -1],
             [0, -1, 0]],
            dtype=torch.float32
        ).view(1, 1, 3, 3)
        self.register_buffer("highpass", kernel)

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AvgPool2d(2),

            nn.AdaptiveAvgPool2d(1)
        )
        self.classifier = nn.Linear(64, 1)

    def forward(self, x):
        # High-pass filter is fixed; only the CNN layers learn.
        x = F.conv2d(x, self.highpass)
        x = torch.clamp(x, -5, 5) / 5.0
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x).squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallSteganalysisCNN().to(device)

with torch.no_grad():
    output = model(images.to(device))

parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Device:", device)
print("Trainable parameters:", parameters)
print("Output shape:", tuple(output.shape))

Device: cuda
Trainable parameters: 23585
Output shape: (4,)


In [ ]:
import copy
import random
import numpy as np
from sklearn.metrics import roc_auc_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Start from a fresh model rather than the one used for the shape check.
model = SmallSteganalysisCNN().to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.0001)

best_val_loss = float("inf")
best_weights = None
epochs_without_improvement = 0

for epoch in range(1, 9):
    model.train()
    train_loss_total = 0.0

    for batch_images, batch_labels in train_loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.float().to(device)

        optimizer.zero_grad()
        logits = model(batch_images)
        loss = loss_fn(logits, batch_labels)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item() * len(batch_labels)

    model.eval()
    val_loss_total = 0.0
    val_labels = []
    val_scores = []

    with torch.no_grad():
        for batch_images, batch_labels in val_loader:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.float().to(device)

            logits = model(batch_images)
            loss = loss_fn(logits, batch_labels)
            scores = torch.sigmoid(logits)

            val_loss_total += loss.item() * len(batch_labels)
            val_labels.extend(batch_labels.cpu().numpy())
            val_scores.extend(scores.cpu().numpy())

    train_loss = train_loss_total / len(train_data)
    val_loss = val_loss_total / len(val_data)
    val_auc = roc_auc_score(val_labels, val_scores)

    print(
        f"Epoch {epoch}: train loss={train_loss:.4f}, "
        f"val loss={val_loss:.4f}, val AUC={val_auc:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement == 3:
            print("Stopped early: validation loss did not improve.")
            break

model.load_state_dict(best_weights)
torch.save(best_weights, "/content/boss_project/cnn_0p4_best.pt")
print("Saved the best validation-loss model.")

Epoch 1: train loss=0.6481, val loss=0.6613, val AUC=0.856
Epoch 2: train loss=0.6158, val loss=0.4534, val AUC=0.903
Epoch 3: train loss=0.5685, val loss=0.4338, val AUC=0.928
Epoch 4: train loss=0.5415, val loss=0.3889, val AUC=0.925
Epoch 5: train loss=0.5322, val loss=0.4320, val AUC=0.929
Epoch 6: train loss=0.5060, val loss=0.3642, val AUC=0.945
Epoch 7: train loss=0.4914, val loss=0.3763, val AUC=0.963
Epoch 8: train loss=0.5100, val loss=0.3618, val AUC=0.932
Saved the best validation-loss model.


In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score
from torch.utils.data import DataLoader
import numpy as np
import csv

test_loader = DataLoader(test_data, batch_size=4, shuffle=False)
model.eval()

def get_predictions(loader):
    labels, scores = [], []

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            logits = model(batch_images.to(device))
            probabilities = torch.sigmoid(logits).cpu().numpy()

            labels.extend(batch_labels.numpy())
            scores.extend(probabilities)

    return np.array(labels), np.array(scores)

val_labels, val_scores = get_predictions(val_loader)
test_labels, test_scores = get_predictions(test_loader)

threshold = np.quantile(val_scores[val_labels == 0], 0.95)
test_predictions = (test_scores > threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    test_labels, test_predictions, labels=[0, 1]
).ravel()

print("Validation threshold:", round(float(threshold), 4))
print("Validation AUC:", round(roc_auc_score(val_labels, val_scores), 3))
print("Test AUC:", round(roc_auc_score(test_labels, test_scores), 3))
print("Test false positives:", f"{fp}/100")
print("Test stego detected:", f"{tp}/100")
print("Test F1:", round(f1_score(test_labels, test_predictions), 3))

output_path = "/content/boss_project/cnn_0p4_test_predictions.csv"
with open(output_path, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["source_filename", "label", "score", "prediction"])
    for (path, _), label, score, prediction in zip(
        test_data.items, test_labels, test_scores, test_predictions
    ):
        writer.writerow([path.name, label, score, prediction])

print("Predictions saved:", output_path)

Validation threshold: 0.7888
Validation AUC: 0.932
Test AUC: 0.931
Test false positives: 2/100
Test stego detected: 64/100
Test F1: 0.771
Predictions saved: /content/boss_project/cnn_0p4_test_predictions.csv


In [ ]:
train_data_01 = ImagePairDataset("train", "0p1")
val_data_01 = ImagePairDataset("validation", "0p1")
test_data_01 = ImagePairDataset("test", "0p1")

train_loader_01 = DataLoader(train_data_01, batch_size=4, shuffle=True)
val_loader_01 = DataLoader(val_data_01, batch_size=4, shuffle=False)

print("0.1 bpp train:", len(train_data_01))
print("0.1 bpp validation:", len(val_data_01))
print("0.1 bpp test:", len(test_data_01))

0.1 bpp train: 600
0.1 bpp validation: 200
0.1 bpp test: 200


In [ ]:
import copy
import random
import numpy as np
from sklearn.metrics import roc_auc_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

model_01 = SmallSteganalysisCNN().to(device)
loss_fn_01 = nn.BCEWithLogitsLoss()
optimizer_01 = torch.optim.AdamW(
    model_01.parameters(), lr=0.001, weight_decay=0.0001
)

best_val_loss_01 = float("inf")
best_weights_01 = None
epochs_without_improvement_01 = 0

for epoch in range(9,21):
    model_01.train()
    train_loss_total = 0.0

    for batch_images, batch_labels in train_loader_01:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.float().to(device)

        optimizer_01.zero_grad()
        logits = model_01(batch_images)
        loss = loss_fn_01(logits, batch_labels)
        loss.backward()
        optimizer_01.step()

        train_loss_total += loss.item() * len(batch_labels)

    model_01.eval()
    val_loss_total = 0.0
    val_labels_01 = []
    val_scores_01 = []

    with torch.no_grad():
        for batch_images, batch_labels in val_loader_01:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.float().to(device)

            logits = model_01(batch_images)
            loss = loss_fn_01(logits, batch_labels)

            val_loss_total += loss.item() * len(batch_labels)
            val_labels_01.extend(batch_labels.cpu().numpy())
            val_scores_01.extend(torch.sigmoid(logits).cpu().numpy())

    train_loss = train_loss_total / len(train_data_01)
    val_loss = val_loss_total / len(val_data_01)
    val_auc = roc_auc_score(val_labels_01, val_scores_01)

    print(
        f"Epoch {epoch}: train loss={train_loss:.4f}, "
        f"val loss={val_loss:.4f}, val AUC={val_auc:.3f}"
    )

    if val_loss < best_val_loss_01:
        best_val_loss_01 = val_loss
        best_weights_01 = copy.deepcopy(model_01.state_dict())
        epochs_without_improvement_01 = 0
    else:
        epochs_without_improvement_01 += 1
        if epochs_without_improvement_01 == 3:
            print("Stopped early: validation loss did not improve.")
            break

model_01.load_state_dict(best_weights_01)
torch.save(best_weights_01, "/content/boss_project/cnn_0p1_best.pt")
print("Saved the best 0.1 bpp model.")

Epoch 9: train loss=0.7000, val loss=0.7004, val AUC=0.545
Epoch 10: train loss=0.6998, val loss=0.6919, val AUC=0.559
Epoch 11: train loss=0.6956, val loss=0.6875, val AUC=0.546
Epoch 12: train loss=0.6967, val loss=0.6885, val AUC=0.563
Epoch 13: train loss=0.6946, val loss=0.6805, val AUC=0.571
Epoch 14: train loss=0.6925, val loss=0.6796, val AUC=0.574
Epoch 15: train loss=0.6908, val loss=0.6784, val AUC=0.610
Epoch 16: train loss=0.6965, val loss=0.6777, val AUC=0.593
Epoch 17: train loss=0.6909, val loss=0.7026, val AUC=0.599
Epoch 18: train loss=0.6873, val loss=0.6718, val AUC=0.591
Epoch 19: train loss=0.6914, val loss=0.6816, val AUC=0.577
Epoch 20: train loss=0.6847, val loss=0.6605, val AUC=0.607
Saved the best 0.1 bpp model.


In [ ]:
def validate_01():
    model_01.eval()
    loss_total = 0.0
    labels = []
    scores = []

    with torch.no_grad():
        for batch_images, batch_labels in val_loader_01:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.float().to(device)

            logits = model_01(batch_images)
            loss_total += (
                loss_fn_01(logits, batch_labels).item()
                * len(batch_labels)
            )
            labels.extend(batch_labels.cpu().numpy())
            scores.extend(torch.sigmoid(logits).cpu().numpy())

    return (
        loss_total / len(val_data_01),
        roc_auc_score(labels, scores)
    )

for actual_epoch in range(13, 17):
    if actual_epoch > 13:  # Epoch 13 training already finished.
        model_01.train()
        train_loss_total = 0.0

        for batch_images, batch_labels in train_loader_01:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.float().to(device)

            optimizer_01.zero_grad()
            logits = model_01(batch_images)
            loss = loss_fn_01(logits, batch_labels)
            loss.backward()
            optimizer_01.step()

            train_loss_total += loss.item() * len(batch_labels)

    val_loss, val_auc = validate_01()
    print(
        f"Actual epoch {actual_epoch}: "
        f"train loss={train_loss_total / len(train_data_01):.4f}, "
        f"val loss={val_loss:.4f}, val AUC={val_auc:.3f}"
    )

    if val_loss < best_val_loss_01:
        best_val_loss_01 = val_loss
        best_weights_01 = copy.deepcopy(model_01.state_dict())
        epochs_without_improvement_01 = 0
    else:
        epochs_without_improvement_01 += 1
        if epochs_without_improvement_01 == 3:
            print("Stopped early.")
            break

model_01.load_state_dict(best_weights_01)
torch.save(best_weights_01, "/content/boss_project/cnn_0p1_best.pt")
print("Saved the best 0.1 bpp model.")

Actual epoch 13: train loss=0.6847, val loss=0.6750, val AUC=0.609
Actual epoch 14: train loss=0.6919, val loss=0.6619, val AUC=0.661
Actual epoch 15: train loss=0.6833, val loss=0.6517, val AUC=0.666
Actual epoch 16: train loss=0.6853, val loss=0.6658, val AUC=0.613
Saved the best 0.1 bpp model.


In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score
import numpy as np
import csv

test_loader_01 = DataLoader(test_data_01, batch_size=4, shuffle=False)
model_01.eval()

def predict_01(loader):
    labels, scores = [], []

    with torch.no_grad():
        for batch_images, batch_labels in loader:
            logits = model_01(batch_images.to(device))
            labels.extend(batch_labels.cpu().numpy())
            scores.extend(torch.sigmoid(logits).cpu().numpy())

    return np.array(labels), np.array(scores)

val_labels_01, val_scores_01 = predict_01(val_loader_01)
test_labels_01, test_scores_01 = predict_01(test_loader_01)

threshold_01 = np.quantile(val_scores_01[val_labels_01 == 0], 0.95)
test_predictions_01 = (test_scores_01 > threshold_01).astype(int)

tn, fp, fn, tp = confusion_matrix(
    test_labels_01, test_predictions_01, labels=[0, 1]
).ravel()

print("Validation threshold:", round(float(threshold_01), 4))
print("Validation AUC:", round(roc_auc_score(val_labels_01, val_scores_01), 3))
print("Test AUC:", round(roc_auc_score(test_labels_01, test_scores_01), 3))
print("Test false positives:", f"{fp}/100")
print("Test stego detected:", f"{tp}/100")
print("Test F1:", round(f1_score(test_labels_01, test_predictions_01), 3))

output_path = "/content/boss_project/cnn_0p1_test_predictions.csv"
with open(output_path, "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["source_filename", "label", "score", "prediction"])
    for (path, _), label, score, prediction in zip(
        test_data_01.items,
        test_labels_01,
        test_scores_01,
        test_predictions_01
    ):
        writer.writerow([path.name, label, score, prediction])

print("Predictions saved:", output_path)

Validation threshold: 0.5991
Validation AUC: 0.666
Test AUC: 0.662
Test false positives: 7/100
Test stego detected: 15/100
Test F1: 0.246
Predictions saved: /content/boss_project/cnn_0p1_test_predictions.csv


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

project = Path("/content/boss_project")
names = [
    "cnn_0p1_best.pt",
    "cnn_0p4_best.pt",
    "cnn_0p1_test_predictions.csv",
    "cnn_0p4_test_predictions.csv",
]

archive = Path("/content/cnn_results.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as zip_file:
    for name in names:
        path = project / name
        assert path.is_file(), f"Missing: {path}"
        zip_file.write(path, name)

print("Saved:", archive)
files.download(str(archive))

Saved: /content/cnn_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>